# Redes Neurais Recorrentes

Sinais de sensores, áudio e texto são **sequências**: a ordem dos elementos importa, e o tamanho pode variar de um exemplo para outro. Uma MLP recebe um vetor de tamanho fixo e não sabe que a posição 10 vem depois da 9.

Uma **rede recorrente** lê a sequência um elemento por vez, sempre com os mesmos pesos, e carrega um **estado oculto** de um passo para o próximo: um resumo de tudo o que ela leu até ali.

Neste material, vemos três camadas recorrentes, a **RNN**, a **LSTM** e a **GRU**, e as usamos para reconhecer atividades físicas pelos sensores do celular e para gerar texto no estilo de Machado de Assis.

In [ ]:
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import ConfusionMatrixDisplay
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## A RNN

Em cada instante $t$, a RNN recebe a entrada $x_t$ e o estado anterior $h_{t-1}$ e calcula o novo estado:

$$h_t = \tanh(W_x x_t + b_x + W_h h_{t-1} + b_h)$$

$W_x$ leva a entrada para o espaço do estado, e $W_h$ mistura o estado anterior. Os mesmos pesos são usados em todos os instantes, como um kernel de convolução é usado em todas as posições da imagem, e por isso a rede aceita sequências de qualquer tamanho. O estado inicial $h_0$ é um vetor de zeros.

Na `nn.RNN`, `input_size` é o tamanho de cada $x_t$ e `hidden_size` é o tamanho de $h_t$. Com `batch_first=True`, a entrada tem formato `[N, T, input_size]`: $N$ sequências de $T$ instantes. A camada devolve o estado de todos os instantes e o estado final.

In [ ]:
rnn = nn.RNN(input_size=4, hidden_size=5, batch_first=True)
x = torch.randn(3, 8, 4)   # 3 sequências de 8 instantes, com 4 valores em cada

outputs, h = rnn(x)
print(outputs.shape)   # [3, 8, 5], o estado em cada instante
print(h.shape)         # [1, 3, 5], o estado final

print(rnn(torch.randn(3, 20, 4))[0].shape)   # [3, 20, 5], a mesma camada lê sequências de 20 instantes

Por dentro, a camada é um laço sobre os instantes, com a equação acima em uma linha. Fazendo a mesma conta com os pesos da `nn.RNN`, chegamos ao mesmo resultado:

In [ ]:
W_x, b_x = rnn.weight_ih_l0, rnn.bias_ih_l0   # [5, 4], [5]
W_h, b_h = rnn.weight_hh_l0, rnn.bias_hh_l0   # [5, 5], [5]

h = torch.zeros(3, 5)   # h_0
for t in range(8):
    h = torch.tanh(x[:, t] @ W_x.T + b_x + h @ W_h.T + b_h)   # [3, 5]

print(torch.allclose(h, outputs[:, -1], atol=1e-6))

### Acessando os estados

O `outputs` guarda o estado de cada sequência em cada instante, e indexamos como qualquer tensor: `outputs[:, t]` é o estado de todas as sequências no instante $t$, e `outputs[:, -1]`, o do último instante. O `h` é esse último estado, com uma dimensão a mais na frente.

In [ ]:
outputs, h = rnn(x)

print(outputs[:, 0].shape)    # [3, 5], o estado de cada sequência no primeiro instante
print(outputs[:, -1].shape)   # [3, 5], o estado de cada sequência no último instante
print(outputs[0].shape)       # [8, 5], todos os estados da primeira sequência
print(torch.allclose(outputs[:, -1], h[0]))

A dimensão a mais do `h` é a das camadas: o formato é `[num_layers, N, hidden_size]`, mesmo com `batch_first=True`. O argumento `num_layers` empilha camadas recorrentes, cada uma lendo a sequência de estados da anterior. Nesse caso, `outputs` traz os estados da última camada, e `h[-1]` é o estado final dela.

O estado final também pode ser uma entrada: passado como segundo argumento, ele toma o lugar do $h_0$ de zeros, e a camada continua a leitura de onde parou. Ler a sequência em dois pedaços leva ao mesmo estado final que ler de uma vez:

In [ ]:
_, h = rnn(x[:, :5])      # lê os 5 primeiros instantes
_, h = rnn(x[:, 5:], h)   # lê os 3 últimos, a partir do estado em que parou

print(torch.allclose(h[0], outputs[:, -1], atol=1e-6))   # o mesmo estado final da leitura de uma vez

## Reconhecimento de atividades

O dataset **UCI HAR** (*Human Activity Recognition*) tem gravações de 30 voluntários com um celular na cintura, fazendo seis atividades: andar, subir escada, descer escada, sentar, ficar em pé e deitar. O celular mede 50 vezes por segundo a aceleração e a rotação nos três eixos, e as gravações foram cortadas em janelas de 2,56 segundos. Cada exemplo é uma sequência de 128 instantes com 9 valores em cada: a aceleração total, a aceleração do corpo (a total sem a gravidade) e o giroscópio, nos eixos $x$, $y$ e $z$. É o que um relógio inteligente faz para contar passos ou detectar uma queda.

O treino tem 21 pessoas e o teste, as outras 9: a rede é avaliada em pessoas que nunca viu. O arquivo tem 60 MB e vem com um zip dentro de outro.

In [ ]:
if not os.path.exists("data/UCI HAR Dataset"):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip", "data/har.zip")
    zipfile.ZipFile("data/har.zip").extractall("data")
    zipfile.ZipFile("data/UCI HAR Dataset.zip").extractall("data")


def load_har(split):
    folder = f"data/UCI HAR Dataset/{split}"
    signals = [f"{sensor}_{axis}" for sensor in ["total_acc", "body_acc", "body_gyro"] for axis in "xyz"]
    x = np.stack([np.loadtxt(f"{folder}/Inertial Signals/{signal}_{split}.txt", dtype=np.float32)
                  for signal in signals], axis=-1)                                   # [N, 128, 9]
    y = np.loadtxt(f"{folder}/y_{split}.txt", dtype=np.int64) - 1                    # [N], de 0 a 5
    return torch.tensor(x), torch.tensor(y)


train_x, train_y = load_har("train")
test_x, test_y = load_har("test")
activities = ["andar", "subir escada", "descer escada", "sentado", "em pé", "deitado"]
print(train_x.shape, test_x.shape)

A aceleração total de uma janela de cada atividade. Andar, subir e descer escada oscilam a cada passo. Nas atividades paradas, o sinal é quase constante, e o que muda é a direção da gravidade: deitado, ela aparece em outro eixo. Sentado e em pé são parecidos, com o celular na mesma posição.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharey=True)
for activity, ax in enumerate(axes.ravel()):
    example = train_x[train_y == activity][0]   # [128, 9]
    ax.plot(example[:, :3])                     # os 3 eixos da aceleração total
    ax.set_title(activities[activity])
axes[0, 0].legend(["x", "y", "z"])
plt.tight_layout()
plt.show()

Os canais têm escalas diferentes, e normalizamos cada um com a média e o desvio padrão do treino, como fazíamos com os pixels.

In [ ]:
mean = train_x.mean(dim=(0, 1))   # [9], um valor por canal
std = train_x.std(dim=(0, 1))
train_x = (train_x - mean) / std
test_x = (test_x - mean) / std
train_dataloader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)

Para classificar uma sequência, usamos o estado do último instante, o resumo da sequência inteira, como entrada de uma camada linear. A `SequenceModel` recebe a camada recorrente pronta, e assim o mesmo modelo serve para a RNN, a LSTM e a GRU.

In [ ]:
class SequenceModel(nn.Module):
    def __init__(self, layer, output_size):
        super().__init__()
        self.layer = layer   # nn.RNN, nn.LSTM ou nn.GRU, com batch_first=True
        self.fc = nn.Linear(layer.hidden_size, output_size)

    def forward(self, x):
        # x: [N, T, input_size]
        outputs, _ = self.layer(x)       # [N, T, hidden_size]
        return self.fc(outputs[:, -1])   # [N, output_size], a partir do último instante

A `train` é a dos materiais anteriores, com o **gradient clipping** do material de estabilidade antes do passo do otimizador: em uma rede recorrente, o gradiente passa pelos mesmos pesos uma vez por instante e pode explodir.

In [ ]:
def train(model, dataloader, epochs, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"[{epoch + 1}/{epochs}] perda {total_loss / len(dataloader):.4f}")


def predict(model, x):
    model.eval()
    with torch.no_grad():
        return model(x.to(device)).cpu()

Treinamos uma RNN com estado de 64 valores:

In [ ]:
torch.manual_seed(42)
rnn_model = SequenceModel(nn.RNN(input_size=9, hidden_size=64, batch_first=True), output_size=6).to(device)
train(rnn_model, train_dataloader, epochs=30)

predictions = predict(rnn_model, test_x).argmax(dim=1)
print(f"acurácia de teste {(predictions == test_y).float().mean():.1%}")

In [ ]:
probs = predict(rnn_model, test_x).softmax(dim=1)   # [N, 6]
correct = (predictions == test_y).nonzero().flatten()
wrong = (predictions != test_y).nonzero().flatten()

torch.manual_seed(0)
examples = torch.cat([correct[torch.randperm(len(correct))[:4]],   # primeira linha: acertos
                      wrong[torch.randperm(len(wrong))[:4]]])      # segunda linha: erros

fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey=True)
for i, ax in zip(examples, axes.ravel()):
    ax.plot(test_x[i, :, :3])   # os 3 eixos da aceleração total, já normalizados
    ax.set_title(f"real: {activities[test_y[i]]}\nprevisto: {activities[predictions[i]]} ({probs[i, predictions[i]]:.0%})",
                 color="green" if predictions[i] == test_y[i] else "red")
axes[0, 0].legend(["x", "y", "z"])
plt.tight_layout()
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(test_y, predictions, display_labels=activities, xticks_rotation=45)
plt.show()

A maior confusão é entre sentado e em pé, que diferem só por uma pequena mudança na inclinação do celular. As três atividades com movimento também se confundem entre si, porque têm o mesmo ritmo de passos.

## A limitação da RNN

Para treinar, o PyTorch desenrola o laço do início da aula: uma sequência de $T$ instantes vira uma rede de $T$ camadas, todas com os mesmos pesos, e o backpropagation percorre essas camadas de trás para frente. É o **backpropagation through time**.

O gradiente que chega ao instante $t$ foi multiplicado por $W_h$ e pela derivada da $\tanh$ uma vez para cada instante depois dele. É o **vanishing gradient** do material de estabilidade, com a profundidade trocada pelo tamanho da sequência: o gradiente dos primeiros instantes chega praticamente nulo, e a RNN aprende a usar só o fim da sequência.

Medimos isso na RNN treinada com o gradiente da perda em relação à entrada de cada instante, que mostra quanto do sinal de erro chega até ali:

In [ ]:
def input_gradient(model, x, y):
    model.train()   # na GPU, o backward da camada recorrente exige o modo de treino
    x = x.to(device).requires_grad_()
    F.cross_entropy(model(x), y.to(device)).backward()
    return x.grad.norm(dim=2).mean(dim=0).cpu()   # [T], a norma média do gradiente em cada instante


plt.figure(figsize=(8, 3))
plt.plot(input_gradient(rnn_model, test_x, test_y))
plt.yscale("log")
plt.xlabel("instante")
plt.ylabel("norma do gradiente")
plt.show()

O eixo é logarítmico: o gradiente do primeiro instante é cerca de mil vezes menor que o do último. Nas janelas de 128 instantes do HAR, o começo da janela quase não participa do treino.

## LSTM

A **LSTM** (*Long Short-Term Memory*) resolve isso com um segundo estado, a **célula** $c_t$, uma memória que atravessa a sequência sem passar por uma multiplicação de matriz a cada instante. Três **portas**, vetores com valores entre 0 e 1 calculados a partir de $x_t$ e $h_{t-1}$, decidem o que acontece com ela:

$$
\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) && \text{porta de esquecimento} \\
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) && \text{porta de entrada} \\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) && \text{porta de saída} \\
\tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) && \text{candidato a memória} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

em que $\sigma$ é a sigmoide e $\odot$ é o produto elemento a elemento. O $\tilde{c}_t$ é a conta da RNN, a informação nova do instante. A porta de esquecimento decide quanto da memória antiga fica, a de entrada decide quanto da informação nova entra, e a de saída decide quanto da memória aparece no estado $h_t$.

O importante é a linha de $c_t$: a memória é atualizada por uma **soma**, como nas conexões residuais da ResNet. Com $f_t$ perto de 1, a memória e o gradiente atravessam muitos instantes sem encolher, e a rede aprende quando vale a pena guardar e quando vale a pena esquecer.

A `nn.LSTM` tem os mesmos argumentos da `nn.RNN` e devolve os dois estados finais, $h$ e $c$. O `outputs` continua sendo o $h_t$ de cada instante, e da célula só temos o valor final:

In [ ]:
lstm = nn.LSTM(input_size=4, hidden_size=5, batch_first=True)
outputs, (h, c) = lstm(x)
print(outputs.shape)       # [3, 8, 5]
print(h.shape, c.shape)    # [1, 3, 5] cada
print(torch.allclose(outputs[:, -1], h[0]))

## GRU

A **GRU** (*Gated Recurrent Unit*) é uma simplificação da LSTM, com um estado só e duas portas:

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1} + b_z) && \text{porta de atualização} \\
r_t &= \sigma(W_r x_t + U_r h_{t-1} + b_r) && \text{porta de reinício} \\
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h) && \text{candidato a estado} \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
\end{aligned}
$$

A porta de atualização faz o papel das portas de esquecimento e de entrada juntas: o novo estado é uma média entre o antigo e o candidato, e com $z_t$ perto de 0 o estado passa adiante sem mudar. A porta de reinício decide quanto do estado anterior entra no cálculo do candidato.

Cada porta tem os próprios pesos, do tamanho dos pesos de uma RNN. A LSTM calcula quatro conjuntos de pesos (três portas e o candidato) e a GRU, três:

In [ ]:
for layer in [nn.RNN(9, 64), nn.LSTM(9, 64), nn.GRU(9, 64)]:
    print(type(layer).__name__, sum(p.numel() for p in layer.parameters()), "parâmetros")

### LSTM e GRU no HAR

Como a `SequenceModel` recebe a camada pronta, basta trocar a `nn.RNN` pela `nn.LSTM` ou pela `nn.GRU`, com os mesmos argumentos. Depois, repetimos o gráfico do gradiente com os três modelos.

In [ ]:
models = {"RNN": rnn_model}
for layer_class in [nn.LSTM, nn.GRU]:
    torch.manual_seed(42)
    model = SequenceModel(layer_class(input_size=9, hidden_size=64, batch_first=True), output_size=6).to(device)
    train(model, train_dataloader, epochs=30)
    predictions = predict(model, test_x).argmax(dim=1)
    print(f"{layer_class.__name__}: acurácia de teste {(predictions == test_y).float().mean():.1%}")
    models[layer_class.__name__] = model

In [ ]:
plt.figure(figsize=(8, 3))
for name, model in models.items():
    plt.plot(input_gradient(model, test_x, test_y), label=name)
plt.yscale("log")
plt.xlabel("instante")
plt.ylabel("norma do gradiente")
plt.legend()
plt.show()

A LSTM e a GRU acertam cerca de 91% das janelas, contra 87% da RNN, com o mesmo tamanho de estado e o mesmo treino, e o gradiente delas chega ao começo da janela na mesma ordem de grandeza do fim. A diferença entre as duas é pequena, o que é comum: a GRU tem menos parâmetros e é um pouco mais rápida, nenhuma das duas é melhor em todos os problemas, e na prática vale testar as duas.

## Gerando texto

Uma rede recorrente também pode **gerar** sequências. Treinamos uma LSTM para prever a próxima letra de um texto a partir das 64 letras anteriores e depois usamos essa previsão, letra a letra, para escrever um texto novo.

O texto é *Dom Casmurro*, de Machado de Assis, do Projeto Gutenberg. Tiramos o cabeçalho e o rodapé em inglês e juntamos as quebras de linha em espaços.

In [ ]:
if not os.path.exists("data/dom_casmurro.txt"):
    urllib.request.urlretrieve("https://gutenberg.pglaf.org/5/5/7/5/55752/55752-0.txt", "data/dom_casmurro.txt")

text = open("data/dom_casmurro.txt", encoding="utf-8").read()
text = text.split("***")[2].split("End of the Project Gutenberg")[0]   # o livro fica entre as linhas *** START *** e *** END ***
text = " ".join(text.split())
print(len(text), "caracteres")
print(text[1000:1300])

Cada caractere vira um número, a sua posição na lista `chars` de caracteres do texto. A ortografia é a de 1899, com "cousa" e "elles".

A `unfold` corta o texto em todas as janelas de 65 caracteres, deslocadas de um em um: os 64 primeiros são a entrada e o último é o alvo, a próxima letra.

In [ ]:
chars = sorted(set(text))
encoded = torch.tensor([chars.index(char) for char in text])   # [376486]
print(len(chars), "caracteres diferentes:", "".join(chars))

windows = encoded.unfold(0, 65, 1)                              # [376422, 65]
text_dataloader = DataLoader(TensorDataset(windows[:, :-1], windows[:, -1]), batch_size=256, shuffle=True)
print(windows.shape)

print("".join(chars[i] for i in windows[1000, :-1]), "->", chars[windows[1000, -1]])   # uma janela e o seu alvo

A rede vê a letra como um vetor **one-hot**, com 1 na posição dela e 0 nas outras. Para não guardar um vetor desses para cada letra de cada janela, o `TextModel` faz a conversão dentro do `forward`, só para o lote. A saída é uma pontuação para cada caractere, e o treino é uma classificação com uma classe por caractere: a entropia cruzada compara as pontuações com a letra que vem de fato no texto.

In [ ]:
class TextModel(nn.Module):
    def __init__(self, num_chars, hidden_size):
        super().__init__()
        self.num_chars = num_chars
        self.layer = nn.LSTM(num_chars, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_chars)

    def forward(self, x):
        # x: [N, T], o índice de cada caractere
        x = F.one_hot(x, self.num_chars).float()   # [N, T, num_chars]
        outputs, _ = self.layer(x)                   # [N, T, hidden_size]
        return self.fc(outputs[:, -1])               # [N, num_chars], a próxima letra


print(TextModel(len(chars), hidden_size=256)(windows[:2, :-1]).shape)   # [2, num_chars], uma pontuação por caractere

São quase 400 mil janelas por época, e cada época leva uns 15 segundos em uma GPU.

In [ ]:
torch.manual_seed(42)
text_model = TextModel(len(chars), hidden_size=256).to(device)
train(text_model, text_dataloader, epochs=5, lr=3e-3)

### Geração autoregressiva

A rede treinada faz uma única coisa: recebe 64 letras e devolve a probabilidade de cada caractere vir em seguida, $p(x_t \mid x_{t-64}, \dots, x_{t-1})$. Para escrever um texto inteiro, usamos essa previsão em um laço. Sorteamos uma letra, acrescentamos no fim do texto, a janela anda uma posição e a rede prevê a próxima letra a partir de uma janela que já inclui o que ela mesma escreveu. Um modelo que gera assim, alimentado pelas próprias saídas, é **autoregressivo**, e a probabilidade que a rede dá ao texto inteiro é o produto das probabilidades de cada letra:

$$p(x_1, \dots, x_n) = \prod_{t} p(x_t \mid x_{t-64}, \dots, x_{t-1})$$

No treino, a janela sempre vem do livro, e a rede nunca vê os próprios erros. Na geração, a janela é feita das letras que a rede sorteou: uma letra errada entra na janela das seguintes, que passam a ser previstas a partir de um texto que não existe no livro, e os desvios se acumulam ao longo do texto. Os modelos de linguagem geram texto do mesmo jeito, com **tokens**, pedaços de palavras, no lugar das letras e janelas de milhares de tokens.

Para escolher a letra, sorteamos o próximo caractere com as probabilidades da softmax. Sempre escolher o mais provável leva a um texto que se repete. A **temperatura** divide as pontuações antes da softmax: abaixo de 1, as letras mais prováveis ficam ainda mais prováveis, e o texto fica mais conservador; acima de 1, as probabilidades se aproximam, e o texto fica mais variado e com mais erros.

In [ ]:
def generate(model, start, length, temperature=0.8):
    result = start
    for _ in range(length):
        window = torch.tensor([[chars.index(char) for char in result[-64:]]])   # [1, 64], as últimas 64 letras
        probs = F.softmax(predict(model, window)[0] / temperature, dim=0)      # [num_chars]
        result += chars[torch.multinomial(probs, 1).item()]                    # a letra sorteada entra no fim do texto
    return result


print(generate(text_model, "Capitu olhou para mim e ", 400))

A rede aprendeu, letra a letra, a escrever palavras do português, com a ortografia antiga, a pontuação e os diálogos, que no livro começam com dois hífens. As frases não fazem sentido: 64 letras são umas dez palavras, e uma LSTM pequena treinada por um minuto não vai muito além da forma do texto.

## Exercícios

### Exercício 1: tamanho e profundidade

No HAR, treine a LSTM com `hidden_size` igual a 16 e a 128, e depois com `num_layers=2`. Compare a acurácia de teste e o número de parâmetros. Algum deles melhora a separação entre sentado e em pé?

### Exercício 2: temperatura e GRU

Gere textos com temperatura 0,2, 0,5, 1 e 1,5 e descreva o que muda. Depois, troque a `nn.LSTM` do `TextModel` por uma `nn.GRU` e compare a perda final e os textos gerados.

### Exercício 3: outro texto

Treine o `TextModel` com outro texto: outro livro do Projeto Gutenberg, letras de músicas, código Python ou as suas mensagens. O que a rede aprende a imitar em cada caso?